# Barranquilla MOPAC: perfil de la parte Python de la fuerza QM/MM

En `qmmm_speed2_a100` (commit 46637c1f, tripsina–BPTI, 0,5 fs, tolerancia 0,01) un paso tomó 42,8 ms: MOPAC 28,5,
**la parte Python de la fuerza 9,0**, largo alcance 2,5 y OpenMM 1,8. Este cuaderno mide en qué se van esos 9 ms, con
MOPAC sustituido por un *stub* instantáneo (no hace falta compilar MOPAC):

| Bloque | Qué hace |
|---|---|
| A | tiempos por paso y por llamada, y perfil línea a línea de `_compute_dev` y del largo alcance |
| B | lo mismo con `CUDA_LAUNCH_BLOCKING=1`: cada kernel síncrono, así cada línea carga con su tiempo de GPU |
| C | `nsys` de 50 llamadas: kernels, copias y sincronizaciones por llamada |

También mide operaciones sueltas con los tamaños de este sistema: lanzar un kernel, `bincount`, `nonzero`, copiar
posiciones y fuerzas, `getState` de OpenMM.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*, y luego *Ejecutar todo* (~5 min).
Usa el estado inicial de la tripsina guardado en Google Drive por los cuadernos anteriores. La última celda descarga
un zip.

## 1. Configuración

In [ ]:
COMMIT = '528881dc'
RESULTS_NAME = 'qmmm_pyprof_a100'
PREP_FROM = ['qmmm_speed2_a100/runs/prep', 'qmmm_dt_a100/runs/C_drift/prep', 'qmmm_guess2_a100/runs/C_drift/prep']
REQUIRE_GPU = 'A100'           # None: aceptar cualquier GPU

## 2. GPU, CPU, Google Drive

In [ ]:
import os, re, subprocess, sys, time, shutil, json
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}. Cambie el tipo de entorno a {REQUIRE_GPU} '
                     f'(o ponga REQUIRE_GPU = None) y vuelva a ejecutar todo.')
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/mopac_paper_bench')
prep = next((BASE / p for p in PREP_FROM if (BASE / p / 'state.xml').exists()), None)
if prep is None:
    raise SystemExit(f'No encuentro el estado inicial de la tripsina en {BASE} ({PREP_FROM})')
print('estado inicial:', prep)
OUT = Path('/content') / RESULTS_NAME
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT, 'prep': str(prep)},
                                         indent=1))

## 3. Código (sin compilar), OpenMM, CuPy, line_profiler

In [ ]:
SRC = Path('/content/mopac_gpu')
shutil.rmtree(SRC, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} line_profiler 2>&1 | tail -2", check=False)
QM = SRC / 'scripts' / 'qmmm'
run(f"cd {QM} && {sys.executable} -c \"import openmm, cupy; from qmmm_pme import gpu_available, fused_far_field_available; "
    f"print('OpenMM', openmm.__version__, 'CuPy', cupy.__version__, 'GPU:', gpu_available(), 'fused:', fused_far_field_available())\"")

## 4. Herramientas

In [ ]:
Path('/content/prof_dev.py').write_text('"""The Python part of the QM/MM force on the device path (CuPy), MOPAC replaced by an instant deterministic stub.\n    python prof_dev.py SCRIPTS_DIR PREP_DIR OUT_DIR [PLATFORM]\nWrites to OUT_DIR: timing.json (step, compute, far field, sizes), lprof.txt (line profile of _compute_dev,\n_image_dev and the far-field class), micro.json (cost of single operations at the sizes of this system).\nCUDA_LAUNCH_BLOCKING=1 in the environment makes every kernel synchronous, so the line profile then charges each\nline with its own GPU time.  PROF_NSYS=N: only N calls between cudaProfilerStart/Stop (for nsys\n--capture-range=cudaProfilerApi), nothing else."""\nimport json\nimport os\nimport sys\nimport time\n\nimport numpy as np\nimport openmm as mm\nimport openmm.app as app\nimport openmm.unit as u\n\nsys.path.insert(0, sys.argv[1])\nimport openmm_mopac as om  # noqa: E402\nimport qmmm_pme as qp  # noqa: E402\nfrom nve_drift import FF, make_system  # noqa: E402\nfrom run_qmmm_md import BACKBONE, parse_residues, residue_matches  # noqa: E402\n\nprep, out = sys.argv[2], sys.argv[3]\nplat = sys.argv[4] if len(sys.argv) > 4 else "CUDA"\nos.makedirs(out, exist_ok=True)\n\n\nclass Stub:\n    def __init__(self, libpath, nums, x0, charge=0, model=0, tolerance=0.01, guess=None):\n        self.n = len(nums)\n        self.scf_iterations = 0\n\n    def step(self, xq, xs, qs, ron, roff, qm_pot=None):\n        xs = np.reshape(xs, (-1, 3))\n        d = xs - np.mean(xq, axis=0)\n        self.mm_pot = 0.1 * np.sin(7.0 * qs + 0.3 * d[:, 0])\n        self.qm_charges = 0.05 * np.cos(np.arange(self.n))\n        return float(np.sum(qs * self.mm_pot)), 0.02 * (xq - xq.mean(axis=0)), 0.01 * d\n\n    def close(self):\n        pass\n\n\nom.QMSession = Stub\nimport cupy as cp  # noqa: E402\n\nsync = cp.cuda.Device().synchronize\nT = {"compute": [], "far": [], "grad": [], "stub": []}\n\n\ndef timed(f, key):\n    def g(*a, **k):\n        t = time.perf_counter()\n        r = f(*a, **k)\n        T[key].append(time.perf_counter() - t)\n        return r\n    return g\n\n\n# before add_qmmm: the PythonForce keeps the bound method it was given\norig = {"compute": om.MopacQMMMForce.compute, "far": om.MopacQMMMForce._pme_far_potential,\n        "grad": om.MopacQMMMForce._pme_far_gradient, "stub": Stub.step}\nom.MopacQMMMForce.compute = timed(orig["compute"], "compute")\nom.MopacQMMMForce._pme_far_potential = timed(orig["far"], "far")\nom.MopacQMMMForce._pme_far_gradient = timed(orig["grad"], "grad")\nStub.step = timed(orig["stub"], "stub")\n\nff = app.ForceField(*FF)\ntop = app.PDBFile(os.path.join(prep, "top.pdb")).topology\nwith open(os.path.join(prep, "state.xml")) as fh:\n    st0 = mm.XmlSerializer.deserialize(fh.read())\nsystem = make_system(ff, top, False)\nqm = [x.index for x in top.atoms() for spec, sc in parse_residues("E:57:sc,E:102:sc,E:195:sc,I:15,I:16")\n      if residue_matches(x.residue, spec) and not (sc and x.name in BACKBONE)]\nqmmm = om.add_qmmm(system, top, st0.getPositions(), qm, 0, "stub", ron=10.0, roff=12.0)\nif qmmm.xp is not cp:\n    raise SystemExit("device path not in use (QMMM_MM_GPU / CuPy / pme_gpu)")\np = mm.Platform.getPlatformByName(plat)\nprops = {"Precision": "mixed"} if plat in ("CUDA", "OpenCL") else {}\ninteg = mm.VerletIntegrator(0.5 * u.femtoseconds)\nctx = mm.Context(system, integ, p, props)\nctx.setPeriodicBoxVectors(*st0.getPeriodicBoxVectors())\nctx.setPositions(st0.getPositions())\nctx.setVelocities(st0.getVelocities())\ninteg.step(5)                                      # grid and kernels set up\nstate = ctx.getState(getPositions=True)\n\nnsys_calls = int(os.environ.get("PROF_NSYS", "0"))\nif nsys_calls:\n    for _ in range(10):\n        qmmm.compute(state)\n    sync()\n    cp.cuda.runtime.profilerStart()\n    for _ in range(nsys_calls):\n        qmmm.compute(state)\n    sync()\n    cp.cuda.runtime.profilerStop()\n    print("nsys range done:", nsys_calls, "calls", flush=True)\n    sys.exit(0)\n\n\n# 1. whole steps and compute()\nn = 200\ninteg.step(5)\nfor k in T:\n    T[k].clear()\nt0 = time.perf_counter()\ninteg.step(n)\nstep_ms = (time.perf_counter() - t0) / n * 1e3\nmed = {k: float(np.median(v) * 1e3) for k, v in T.items()}\nres = {"platform": plat, "blocking": os.environ.get("CUDA_LAUNCH_BLOCKING", ""), "atoms": system.getNumParticles(),\n       "qm_atoms": len(qm), "links": len(qmmm.link_q), "mm_charges": len(qmmm.mm_idx_all),\n       "groups": len(qmmm.group_ids), "n_mm_switched": qmmm.last["n_mm"],\n       "far_class": type(qmmm.far).__name__, "step_ms": step_ms, "compute_ms": med["compute"],\n       "far_ms": med["far"] + med["grad"], "stub_ms": med["stub"],\n       "python_ms": med["compute"] - med["far"] - med["grad"] - med["stub"],\n       "outside_compute_ms": step_ms - float(np.mean(T["compute"]) * 1e3),\n       "cupy": cp.__version__, "openmm": mm.__version__}\nfor k in T:\n    T[k].clear()\nfor _ in range(100):\n    qmmm.compute(state)\nres["compute_same_state_ms"] = float(np.median(T["compute"]) * 1e3)\nprint(json.dumps(res, indent=1), flush=True)\nwith open(os.path.join(out, "timing.json"), "w") as fh:\n    json.dump(res, fh, indent=1)\n\n# 2. line profile\nom.MopacQMMMForce.compute = orig["compute"]\nom.MopacQMMMForce._pme_far_potential, om.MopacQMMMForce._pme_far_gradient = orig["far"], orig["grad"]\nStub.step = orig["stub"]\ntry:\n    from line_profiler import LineProfiler\n    lp = LineProfiler()\n    om.MopacQMMMForce._compute_dev = lp(om.MopacQMMMForce._compute_dev)\n    om.MopacQMMMForce._image_dev = lp(om.MopacQMMMForce._image_dev)\n    om.MopacQMMMForce._check_box = lp(om.MopacQMMMForce._check_box)\n    for name in ("potential", "gradient", "_grid_potential", "_gather"):\n        setattr(qp.FarFieldGPU, name, lp(getattr(qp.FarFieldGPU, name)))\n    for _ in range(100):\n        qmmm.compute(state)\n    sync()\n    with open(os.path.join(out, "lprof.txt"), "w") as fh:\n        lp.print_stats(stream=fh, output_unit=1e-3, stripzeros=True)\n    print("line profile:", os.path.join(out, "lprof.txt"), flush=True)\nexcept ImportError:\n    print("line_profiler not installed: no line profile", flush=True)\n\n\n# 3. single operations at this system\'s sizes (microseconds per call, synchronized at the end of each batch)\ndef bench(f, reps=200):\n    f()\n    sync()\n    t = time.perf_counter()\n    for _ in range(reps):\n        f()\n    sync()\n    return (time.perf_counter() - t) / reps * 1e6\n\n\nnm, ng, nq = len(qmmm.mm_idx_all), len(qmmm.group_ids), len(qmmm.qm_idx)\nnsel = qmmm.last["n_mm"]\npos_nm = state.getPositions(asNumpy=True).value_in_unit(u.nanometer)\nd_pos = cp.asarray(pos_nm)\nsmall = cp.zeros(1000)\na_mm = cp.random.random(nm)\nmask_g = cp.random.random(ng) < 0.1\nnear = int(mask_g.sum())\npA = cp.random.random((near, nq))\nuu = cp.random.random((near, nq, 3))\nkern = cp.RawKernel(r\'extern "C" __global__ void nop(double *x) {}\', "nop")\ngof = qmmm.d_group_of\nf_all = cp.zeros((pos_nm.shape[0], 3))\nmicro = {\n    "sizes": dict(atoms=int(pos_nm.shape[0]), mm=nm, groups=ng, near_groups_test=near, qm=nq, switched=nsel),\n    "getState_positions": bench(lambda: ctx.getState(getPositions=True), 50),\n    "getPositions_asNumpy_nm": bench(lambda: state.getPositions(asNumpy=True).value_in_unit(u.nanometer), 50),\n    "getPeriodicBoxVectors": bench(lambda: state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(u.nanometer)),\n    "cp_asarray_positions": bench(lambda: cp.asarray(pos_nm)),\n    "cp_asnumpy_forces": bench(lambda: cp.asnumpy(f_all)),\n    "cp_asarray_3_doubles": bench(lambda: cp.asarray(np.ones(3))),\n    "cp_elementwise_small": bench(lambda: small * 2.0, 2000),\n    "cp_elementwise_mm": bench(lambda: a_mm * 2.0, 2000),\n    "cp_gather_mm": bench(lambda: d_pos[qmmm.d_mm_idx]),\n    "cp_bincount_weights_groups": bench(lambda: cp.bincount(gof, weights=a_mm, minlength=ng)),\n    "cp_nonzero_groups": bench(lambda: cp.nonzero(mask_g)[0]),\n    "cp_bool_index_groups": bench(lambda: a_mm[:ng][mask_g]),\n    "cp_norm_rows": bench(lambda: cp.linalg.norm(uu, axis=2)),\n    "cp_einsum_ga_gak": bench(lambda: cp.einsum("ga,gak->gk", pA, uu)),\n    "cp_searchsorted": bench(lambda: cp.searchsorted(cp.arange(near), gof[:nsel] % max(near, 1))),\n    "rawkernel_empty_launch": bench(lambda: kern((1,), (32,), (small,)), 2000),\n    "sync_only": bench(sync, 2000),\n}\nprint(json.dumps(micro, indent=1), flush=True)\nwith open(os.path.join(out, "micro.json"), "w") as fh:\n    json.dump(micro, fh, indent=1)\n')
Path('/content/analyze_prof.py').write_text('import re\nimport sqlite3\nfrom collections import defaultdict\n\nimport numpy as np\n\nRX_SEC = re.compile(r\'\\[PROFILE\\] MOZYME_SECTION name=(\\S+) calls=(\\d+) ms=([0-9.]+)\')\nRX_STAGE = re.compile(r\'\\[PROFILE\\] MOZYME_RESIDENT_STAGE name=(\\S+) calls=(\\d+) ms=([0-9.]+)\')\nRX_TRACE = re.compile(r\'\\[MOZYME GPU SCF trace\\] diagg2 staged=(\\d+) nij=(\\d+) rotated=(\\d+) max_chain=(\\d+) \'\n                      r\'lock_spins=(\\d+) nrej=(-?\\d+) lpt=(\\d+) grid=(\\d+) span_us=([0-9.]+) wait_us=([0-9.]+) \'\n                      r\'rotate_us=([0-9.]+)\')\n\n\ndef steady(T, skip=3):\n    """per-call medians (ms) after the first `skip` calls (the first SCF starts from the Lewis structure)"""\n    c, t0, m = np.array(T[\'compute\']), np.array(T[\'t0\']), np.array(T[\'mopac\'])\n    n = len(c)\n    far = (np.array(T[\'pme_far\']) + np.array(T[\'pme_grad\'])) if len(T[\'pme_far\']) == n else np.zeros(n)\n    s = slice(skip, n)\n    gap = t0[skip + 1:] - (t0[skip:-1] + c[skip:-1])\n    it = [x for x in T[\'iters\'][skip:] if x is not None]\n    r = dict(calls=n, step_ms=(t0[-1] + c[-1] - t0[skip]) / (n - skip) * 1e3,\n             compute_ms=np.median(c[s]) * 1e3, mopac_ms=np.median(m[s]) * 1e3, far_ms=np.median(far[s]) * 1e3,\n             python_ms=np.median(c[s] - m[s] - far[s]) * 1e3, gap_ms=np.median(gap) * 1e3,\n             gap_mean_ms=np.mean(gap) * 1e3, gaps_over_50ms=int(np.sum(gap > 0.05)),\n             scf_iterations=float(np.mean(it)) if it else None, mopac_ms_per_iteration=None,\n             first_call_ms=c[0] * 1e3, first_heat=T[\'heat\'][0], first_iters=T[\'iters\'][0])\n    if it:\n        r[\'mopac_ms_per_iteration\'] = r[\'mopac_ms\'] / np.mean(it)\n    return {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in r.items()}\n\n\ndef sections(text):\n    """MOPAC\'s cumulative section times (MOPAC_API_OUTPUT): the last value of each section"""\n    sec = {}\n    for m in RX_SEC.finditer(text):\n        sec[m.group(1)] = (int(m.group(2)), float(m.group(3)))\n    return sec\n\n\ndef stages(text):\n    """resident SCF stages (stdout), summed over the SCF calls"""\n    tot = defaultdict(lambda: [0, 0.0])\n    for m in RX_STAGE.finditer(text):\n        tot[m.group(1)][0] += int(m.group(2))\n        tot[m.group(1)][1] += float(m.group(3))\n    return dict(tot)\n\n\ndef trace(text):\n    """diagg2 sweeps (MOPAC_MOZYME_SCF_TRACE=1): medians over the sweeps"""\n    rows = [tuple(float(x) for x in m.groups()) for m in RX_TRACE.finditer(text)]\n    if not rows:\n        return {}\n    a = np.array(rows)\n    names = [\'staged\', \'nij\', \'rotated\', \'max_chain\', \'lock_spins\', \'nrej\', \'lpt\', \'grid\', \'span_us\', \'wait_us\',\n             \'rotate_us\']\n    r = {\'sweeps\': len(rows)}\n    r.update({n: float(np.median(a[:, i])) for i, n in enumerate(names)})\n    r[\'us_per_rotation_on_critical_path\'] = float(np.median(a[:, 8] / np.maximum(a[:, 3], 1)))\n    return r\n\n\ndef _union(iv):\n    iv = sorted(iv)\n    tot, cs, ce = 0, None, None\n    for s, e in iv:\n        if cs is None or s > ce:\n            if cs is not None:\n                tot += ce - cs\n            cs, ce = s, e\n        else:\n            ce = max(ce, e)\n    if cs is not None:\n        tot += ce - cs\n    return tot\n\n\ndef nsys_summary(sqlite_path, ncalls):\n    """GPU busy time against wall time over the captured range, launches and API calls per QM/MM call"""\n    con = sqlite3.connect(sqlite_path)\n    tables = {r[0] for r in con.execute("select name from sqlite_master where type=\'table\'")}\n    names = dict(con.execute(\'select id, value from StringIds\'))\n    k = con.execute(\'select start, end, shortName from CUPTI_ACTIVITY_KIND_KERNEL\').fetchall()\n    mc = con.execute(\'select start, end from CUPTI_ACTIVITY_KIND_MEMCPY\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_MEMCPY\' in tables else []\n    ms = con.execute(\'select start, end from CUPTI_ACTIVITY_KIND_MEMSET\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_MEMSET\' in tables else []\n    rt = con.execute(\'select start, end, nameId from CUPTI_ACTIVITY_KIND_RUNTIME\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_RUNTIME\' in tables else []\n    t_lo = min([r[0] for r in rt] + [r[0] for r in k])\n    t_hi = max([r[1] for r in rt] + [r[1] for r in k])\n    wall = (t_hi - t_lo) * 1e-6\n    busy = _union([(s, e) for s, e, _ in k] + mc + ms) * 1e-6\n    kern = defaultdict(lambda: [0, 0.0])\n    for s, e, nid in k:\n        kern[names.get(nid, str(nid))][0] += 1\n        kern[names.get(nid, str(nid))][1] += (e - s) * 1e-6\n    api = defaultdict(lambda: [0, 0.0])\n    for s, e, nid in rt:\n        api[names.get(nid, str(nid))][0] += 1\n        api[names.get(nid, str(nid))][1] += (e - s) * 1e-6\n    n = max(1, ncalls)\n    return dict(calls=ncalls, wall_ms_per_call=wall / n, gpu_busy_ms_per_call=busy / n,\n                kernel_ms_per_call=sum(v[1] for v in kern.values()) / n,\n                kernels_per_call=len(k) / n, memcpy_per_call=len(mc) / n, memset_per_call=len(ms) / n,\n                api_calls_per_call=len(rt) / n,\n                top_kernels=[(name, c / n, t / n) for name, (c, t) in sorted(kern.items(), key=lambda x: -x[1][1])[:30]],\n                top_api=[(name, c / n, t / n) for name, (c, t) in sorted(api.items(), key=lambda x: -x[1][1])[:20]])\n')

sys.path.insert(0, '/content')
import analyze_prof as AP
PD = '/content/prof_dev.py'

## A y B. Perfil normal y con `CUDA_LAUNCH_BLOCKING=1`

In [ ]:
for name, envx in (('A_normal', {}), ('B_blocking', {'CUDA_LAUNCH_BLOCKING': '1'})):
    d = OUT / name
    d.mkdir(exist_ok=True)
    run(f'{sys.executable} -u {PD} {QM} {prep} {d} CUDA', env=envx, check=False, log=d / 'log.txt')
    print(f'--- {name}: líneas más caras de _compute_dev')
    if (d / 'lprof.txt').exists():
        txt = (d / 'lprof.txt').read_text()
        part = txt.split('Function: MopacQMMMForce._compute_dev')[1].split('Total time')[0] if '_compute_dev' in txt else ''
        rows = [l for l in part.splitlines() if re.match(r'\s*\d+\s+\d+\s+[\d.]+', l)]
        rows.sort(key=lambda l: -float(l.split()[2]))
        print('\n'.join(rows[:15]))

## C. `nsys` de 50 llamadas

In [ ]:
import glob
NSYS = shutil.which('nsys') or next(iter(sorted(glob.glob('/usr/local/cuda*/bin/nsys') +
                                                glob.glob('/opt/nvidia/nsight-systems*/**/bin/nsys', recursive=True))), None)
print('nsys:', NSYS)
dC = OUT / 'C_nsys'
dC.mkdir(exist_ok=True)
if NSYS:
    cmd = (f'{NSYS} profile -t cuda,nvtx -s none --cpuctxsw=none --capture-range=cudaProfilerApi '
           f'--capture-range-end=stop -o {dC}/rep -f true {sys.executable} -u {PD} {QM} {prep} {dC} CUDA')
    (dC / 'cmd.txt').write_text(cmd)
    with open(dC / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, cwd=dC, env=dict(os.environ, PROF_NSYS='50'), stdout=fh,
                            stderr=subprocess.STDOUT).returncode
    rep = next(dC.glob('rep*.nsys-rep'), None)
    print('rc', rc, 'report', rep)
    if rep:
        subprocess.run(f'{NSYS} stats -r cuda_api_sum,cuda_gpu_kern_sum,cuda_gpu_mem_time_sum -f csv -o {dC}/stats {rep}',
                       shell=True, stdout=open(dC / 'stats_log.txt', 'w'), stderr=subprocess.STDOUT)
        sq = next(dC.glob('rep*.sqlite'), None)
        try:
            s = AP.nsys_summary(str(sq), 50)
            (dC / 'nsys_summary.json').write_text(json.dumps(s, indent=1))
            print({k: v for k, v in s.items() if not k.startswith('top_')})
            for name, c, t in s['top_kernels'][:15]:
                print(f'  {t:8.3f} ms/llamada  {c:7.1f} veces/llamada  {name[:90]}')
            for name, c, t in s['top_api'][:15]:
                print(f'  {t:8.3f} ms/llamada  {c:7.1f} veces/llamada  {name[:90]}')
        except Exception as e:
            print('sqlite:', repr(e))
        for f in list(dC.glob('rep*.sqlite')) + list(dC.glob('rep*.nsys-rep')):
            f.unlink()

## D. Resumen y descarga

In [ ]:
for name in ('A_normal', 'B_blocking'):
    for f in ('timing.json', 'micro.json'):
        p = OUT / name / f
        print(name, f, p.read_text() if p.exists() else 'sin resultado')
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', OUT.parent, OUT.name)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)